# Clase 8 — Decisiones autónomas y aprendizaje por refuerzo

Hasta ahora una consulta produjo una decisión. En un entorno dinámico, las acciones cambian el estado y afectan resultados futuros.

Simularemos la gestión de un ticket. Compararemos una política fija con Q-learning. El LLM no participa del entrenamiento: son tecnologías diferentes.

## Objetivos

- Definir estado, acción, recompensa, política y episodio.
- Construir un simulador controlado.
- Evaluar una política fija.
- Entrenar Q-learning y visualizar su progreso.
- Detectar conductas producidas por una recompensa mal diseñada.

---
## 1. El entorno de tickets

Estados:

- nuevo: ticket recién recibido;
- diagnosticado: ya conocemos categoría y prioridad;
- escalado: fue enviado a una persona;
- resuelto: terminó correctamente;
- abandonado: terminó sin solución.

Acciones: diagnosticar, responder, escalar o esperar.

In [ ]:
import random, numpy as np, pandas as pd
import matplotlib.pyplot as plt

ESTADOS=["nuevo","diagnosticado","escalado","resuelto","abandonado"]
ACCIONES=["diagnosticar","responder","escalar","esperar"]
TERMINALES={"resuelto","abandonado"}

def paso(estado,accion):
    if estado in TERMINALES:
        return estado,0,True
    if estado=="nuevo":
        if accion=="diagnosticar": return "diagnosticado",2,False
        if accion=="escalar": return "escalado",-1,False
        if accion=="responder": return "abandonado",-8,True
        return "nuevo",-1,False
    if estado=="diagnosticado":
        if accion=="responder": return "resuelto",10,True
        if accion=="escalar": return "escalado",3,False
        return estado,-2,False
    if estado=="escalado":
        if accion=="responder": return "resuelto",7,True
        if accion=="esperar": return "escalado",-1,False
        return estado,-2,False

for accion in ACCIONES:
    print("nuevo +",accion,"→",paso("nuevo",accion))

### Cómo leer la transición

paso devuelve nuevo estado, recompensa y si el episodio terminó. La recompensa no es dinero: es una señal numérica diseñada por nosotros.

---
## 2. Política fija

Una política indica qué acción elegir en cada estado. Comenzamos con reglas razonables y medimos su recompensa acumulada.

In [ ]:
POLITICA_FIJA={
 "nuevo":"diagnosticar",
 "diagnosticado":"responder",
 "escalado":"responder",
}

def ejecutar_episodio(politica,max_pasos=10):
    estado="nuevo"; total=0; traza=[]
    for _ in range(max_pasos):
        accion=politica.get(estado,"esperar")
        siguiente,premio,termino=paso(estado,accion)
        traza.append({"estado":estado,"accion":accion,
                      "premio":premio,"siguiente":siguiente})
        total+=premio; estado=siguiente
        if termino: break
    return {"recompensa":total,"estado_final":estado,"traza":traza}

ejecutar_episodio(POLITICA_FIJA)

---
## 3. Explorar y aprender

Q-learning guarda un valor para cada combinación estado–acción. Durante el entrenamiento alterna:

- exploración: probar una acción;
- explotación: elegir la mejor conocida.

La actualización aproxima recompensa actual más valor futuro.

In [ ]:
def crear_q():
    return {e:{a:0.0 for a in ACCIONES} for e in ESTADOS}

def entrenar_q(episodios=600,alpha=0.2,gamma=0.9,epsilon=0.25,
              semilla=42):
    random.seed(semilla)
    q=crear_q(); recompensas=[]
    for _ in range(episodios):
        estado="nuevo"; total=0
        for _ in range(15):
            if random.random()<epsilon:
                accion=random.choice(ACCIONES)
            else:
                accion=max(q[estado],key=q[estado].get)
            siguiente,premio,termino=paso(estado,accion)
            q[estado][accion]+=alpha*(premio+gamma*max(q[siguiente].values())
                                      -q[estado][accion])
            total+=premio; estado=siguiente
            if termino: break
        recompensas.append(total)
    return q,recompensas

Q,recompensas=entrenar_q()
pd.DataFrame(Q).T.round(2)

### Cómo leer la tabla Q

Cada fila es un estado y cada columna una acción. El valor mayor indica la acción preferida según las recompensas observadas. No es una probabilidad ni una explicación causal.

In [ ]:
politica_aprendida={e:max(Q[e],key=Q[e].get)
                    for e in ESTADOS if e not in TERMINALES}
print(politica_aprendida)
print(ejecutar_episodio(politica_aprendida))

---
## 4. ¿Aprendió de verdad?

Una curva por episodio es ruidosa porque existe exploración. Graficamos promedio móvil para observar tendencia.

In [ ]:
serie=pd.Series(recompensas)
promedio=serie.rolling(40,min_periods=1).mean()
plt.figure(figsize=(10,4))
plt.plot(recompensas,alpha=0.2,label="episodio")
plt.plot(promedio,label="promedio móvil",linewidth=2)
plt.xlabel("Episodio"); plt.ylabel("Recompensa")
plt.title("Entrenamiento de Q-learning")
plt.legend(); plt.grid(alpha=0.2); plt.show()

---
## 5. Recompensas mal diseñadas

Si premiamos mucho escalar, el agente puede escalar todo aunque sea innecesario. Eso se llama especificación incorrecta del objetivo: el algoritmo optimiza lo escrito, no nuestra intención.

In [ ]:
def paso_mala_recompensa(estado,accion):
    siguiente,premio,termino=paso(estado,accion)
    if accion=="escalar":
        premio+=20
    return siguiente,premio,termino

print("Diseño original:",paso("nuevo","escalar"))
print("Diseño defectuoso:",paso_mala_recompensa("nuevo","escalar"))

---
## 6. Relación con nuestros agentes

El clasificador o LLM interpreta la consulta. La política decide qué hacer a través del tiempo. No conviene pedir al LLM que finja haber aprendido por refuerzo.

    entrada → clasificación → estado → política → acción → nuevo estado

Cada componente requiere una evaluación diferente.

---
## 📝 Actividad 1 — Cambiar una recompensa

Elegí una consecuencia: demora, escalamiento innecesario o respuesta sin diagnóstico. Cambiá un solo premio, reentrená y compará política y recompensa media.

In [ ]:
# TODO: copiá la función paso y modificá una recompensa.
# Luego adaptá entrenar_q para usarla.
experimento={
 "recompensa_modificada":"...",
 "politica_antes":politica_aprendida,
 "politica_despues":"...",
 "interpretacion":"...",
}
experimento

---
## 📝 Actividad 2 — Exploración vs. explotación

Entrená con epsilon 0.0, 0.1, 0.5 y 1.0. Compará recompensa final y política. Explicá por qué explorar nada y explorar siempre pueden fallar.

In [ ]:
comparacion=[]
for epsilon in [0.0,0.1,0.5,1.0]:
    q,r=entrenar_q(epsilon=epsilon)
    politica={e:max(q[e],key=q[e].get)
              for e in ESTADOS if e not in TERMINALES}
    comparacion.append({"epsilon":epsilon,
      "recompensa_ultimos_100":round(float(np.mean(r[-100:])),2),
      "politica":str(politica)})
pd.DataFrame(comparacion)

---
## 📝 Actividad 3 — Auditoría de la política

Para cada estado, indicá acción aprendida, consecuencia, riesgo y si exigirías supervisión humana.

In [ ]:
auditoria=[]
for estado,accion in politica_aprendida.items():
    auditoria.append({"estado":estado,"accion":accion,
                      "consecuencia":"TODO","riesgo":"TODO",
                      "supervision":"TODO"})
pd.DataFrame(auditoria)

---
## ✅ Resumen

Construimos un entorno, medimos una política fija, entrenamos Q-learning y observamos el efecto de las recompensas. Una política óptima para una señal defectuosa sigue siendo una mala política.

En la Clase 9 evaluaremos juntos reglas, LLM, herramientas y controles dentro de un SGIA.